# Task 4 — Optimize Portfolio Based on Forecast
**Objective:** Use Modern Portfolio Theory (MPT) to construct an optimal portfolio using TSLA forecast returns and historical BND/SPY returns.

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.append('../src')

from data_loader            import load_cleaned
from portfolio_optimization import (
    get_expected_returns, compute_cov_matrix, plot_cov_heatmap,
    simulate_efficient_frontier, max_sharpe_portfolio,
    min_volatility_portfolio, plot_efficient_frontier,
    print_portfolio_summary, portfolio_performance
)
from backtesting import (
    get_backtest_returns, simulate_portfolio,
    compute_performance_metrics, plot_cumulative_returns,
    plot_drawdown, print_metrics_table, BENCHMARK_WEIGHTS
)

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.4f}'.format)
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 2. Load Data

In [ ]:
tickers = ['TSLA', 'BND', 'SPY']
dfs = {ticker: load_cleaned(ticker) for ticker in tickers}

for ticker in tickers:
    print(f"{ticker}: {dfs[ticker].shape}  |  {dfs[ticker].index.min().date()} to {dfs[ticker].index.max().date()}")

## 3. Prepare Expected Returns
- **TSLA**: Use annualized return from the best-performing model forecast (Task 3)
- **BND / SPY**: Use historical mean daily return × 252

In [ ]:
# Load ARIMA future forecast saved in Task 3
arima_fc = pd.read_csv(f'{PROCESSED_DIR}/arima_future_forecast.csv',
                        index_col=0, parse_dates=True)

# Compute TSLA forecasted annualized return
tsla_last_price   = dfs['TSLA']['Adj Close'].iloc[-1]
tsla_forecast_end = arima_fc['forecast'].iloc[-1]
tsla_forecast_ann_return = (tsla_forecast_end / tsla_last_price) - 1

print(f"TSLA last known price    : ${tsla_last_price:.2f}")
print(f"TSLA 12-month forecast   : ${tsla_forecast_end:.2f}")
print(f"TSLA forecasted ann. return: {tsla_forecast_ann_return*100:.2f}%")

In [ ]:
expected_returns = get_expected_returns(dfs, tsla_forecast_ann_return)
print("\nExpected Annual Returns:")
for ticker, ret in expected_returns.items():
    print(f"  {ticker}: {ret*100:.2f}%")

## 4. Compute Covariance Matrix

In [ ]:
cov_matrix = compute_cov_matrix(dfs)
print("Annualized Covariance Matrix:")
print(cov_matrix.to_string())
plot_cov_heatmap(cov_matrix)

## 5. Generate the Efficient Frontier

In [ ]:
print("Simulating 10,000 random portfolios...")
frontier_df = simulate_efficient_frontier(expected_returns, cov_matrix, n=10_000)
print(f"Frontier shape: {frontier_df.shape}")
frontier_df.describe()

## 6. Identify Key Portfolios

In [ ]:
ms_weights, ms_ret, ms_vol, ms_shrp = max_sharpe_portfolio(expected_returns, cov_matrix)
mv_weights, mv_ret, mv_vol, mv_shrp = min_volatility_portfolio(expected_returns, cov_matrix)

print_portfolio_summary('Maximum Sharpe Ratio Portfolio', ms_weights, ms_ret, ms_vol, ms_shrp)
print_portfolio_summary('Minimum Volatility Portfolio',   mv_weights, mv_ret, mv_vol, mv_shrp)

In [ ]:
plot_efficient_frontier(
    frontier_df,
    max_sharpe = (ms_weights, ms_ret, ms_vol, ms_shrp),
    min_vol    = (mv_weights, mv_ret, mv_vol, mv_shrp)
)

## 7. Recommend Optimal Portfolio

In [ ]:
# Select Max Sharpe as the recommended portfolio
recommended_weights = ms_weights
rec_ret, rec_vol, rec_shrp = ms_ret, ms_vol, ms_shrp

print_portfolio_summary('RECOMMENDED PORTFOLIO (Max Sharpe)', recommended_weights, rec_ret, rec_vol, rec_shrp)

# Pie chart of recommended weights
fig, ax = plt.subplots(figsize=(7, 5))
colors  = ['#E31937', '#1F77B4', '#2CA02C']
labels  = [f"{t}\n{w*100:.1f}%" for t, w in recommended_weights.items()]
ax.pie(list(recommended_weights.values()), labels=labels, colors=colors,
       autopct='%1.1f%%', startangle=140, textprops={'fontsize': 11})
ax.set_title('Recommended Portfolio Allocation (Max Sharpe)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROCESSED_DIR}/recommended_portfolio_weights.png', dpi=150, bbox_inches='tight')
plt.show()

### Portfolio Selection Justification

The **Maximum Sharpe Ratio Portfolio** is selected as the recommended allocation because it maximizes risk-adjusted return — delivering the highest excess return per unit of risk taken. Given GMF Investments' mandate to optimize portfolio performance while managing downside risk, the Sharpe-optimal portfolio represents the most efficient use of capital across the three assets. The allocation reflects the model's view on TSLA's forecasted return while anchoring portfolio stability through BND's low-volatility income and SPY's broad market diversification. The Minimum Volatility Portfolio is noted as an alternative for risk-averse clients who prioritize capital preservation over return maximization.

---
# Task 5 — Strategy Backtesting
**Objective:** Validate the optimal portfolio strategy against a 60/40 benchmark over the backtest period (2025–2026).

## 8. Define Backtest Period and Benchmark

In [ ]:
BACKTEST_START = '2025-01-01'

daily_returns = get_backtest_returns(dfs, start=BACKTEST_START)
print(f"Backtest period: {daily_returns.index[0].date()} to {daily_returns.index[-1].date()}")
print(f"Trading days   : {len(daily_returns)}")
print(f"\nBenchmark weights: {BENCHMARK_WEIGHTS}")
print(f"Strategy weights : {recommended_weights}")

## 9. Simulate Strategy and Benchmark

In [ ]:
# Buy-and-hold simulation
strategy_cum_bh  = simulate_portfolio(daily_returns, recommended_weights, rebalance_monthly=False)
benchmark_cum_bh = simulate_portfolio(daily_returns, BENCHMARK_WEIGHTS,   rebalance_monthly=False)

# Monthly rebalancing simulation
strategy_cum_rb  = simulate_portfolio(daily_returns, recommended_weights, rebalance_monthly=True)
benchmark_cum_rb = simulate_portfolio(daily_returns, BENCHMARK_WEIGHTS,   rebalance_monthly=True)

print("Simulations complete.")
print(f"Strategy final value (buy-hold)  : {strategy_cum_bh.iloc[-1]:.4f}")
print(f"Benchmark final value (buy-hold) : {benchmark_cum_bh.iloc[-1]:.4f}")

## 10. Visualize Cumulative Returns

In [ ]:
# Buy-and-hold comparison
plot_cumulative_returns(strategy_cum_bh, benchmark_cum_bh)

In [ ]:
# Monthly rebalancing comparison
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(strategy_cum_rb.index,  strategy_cum_rb.values,
        color='green',     linewidth=2.0, label='Strategy (Monthly Rebalanced)')
ax.plot(benchmark_cum_rb.index, benchmark_cum_rb.values,
        color='steelblue', linewidth=2.0, linestyle='--', label='Benchmark (Monthly Rebalanced)')
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1)
ax.set_title('Backtest: Monthly Rebalancing — Cumulative Returns', fontsize=14, fontweight='bold')
ax.set_ylabel('Portfolio Value (Starting = 1.0)')
ax.set_xlabel('Date')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{PROCESSED_DIR}/backtest_rebalanced.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Drawdown comparison
plot_drawdown(strategy_cum_bh, benchmark_cum_bh)

## 11. Performance Metrics

In [ ]:
strategy_metrics  = compute_performance_metrics(strategy_cum_bh,  label='Optimal Strategy (Buy-Hold)')
benchmark_metrics = compute_performance_metrics(benchmark_cum_bh, label='Benchmark 60/40 (Buy-Hold)')
metrics_df = print_metrics_table(strategy_metrics, benchmark_metrics)

In [ ]:
# Rebalanced metrics
strategy_rb_metrics  = compute_performance_metrics(strategy_cum_rb,  label='Optimal Strategy (Rebalanced)')
benchmark_rb_metrics = compute_performance_metrics(benchmark_cum_rb, label='Benchmark 60/40 (Rebalanced)')
print_metrics_table(strategy_rb_metrics, benchmark_rb_metrics)

In [ ]:
# Metrics bar chart
metrics_plot = pd.DataFrame([strategy_metrics, benchmark_metrics]).set_index('Portfolio')
plot_cols = ['Total Return (%)', 'Ann. Return (%)', 'Sharpe Ratio', 'Max Drawdown (%)']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, plot_cols):
    bars = ax.bar(metrics_plot.index, metrics_plot[col],
                  color=['green', 'steelblue'], edgecolor='white', width=0.5)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
plt.suptitle('Strategy vs Benchmark — Performance Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROCESSED_DIR}/backtest_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Conclusion and Reflection

In [ ]:
strat_sharpe = strategy_metrics['Sharpe Ratio']
bench_sharpe = benchmark_metrics['Sharpe Ratio']
strat_ret    = strategy_metrics['Total Return (%)']
bench_ret    = benchmark_metrics['Total Return (%)']
outperformed = strat_sharpe > bench_sharpe

print(f"Strategy Sharpe  : {strat_sharpe:.4f}")
print(f"Benchmark Sharpe : {bench_sharpe:.4f}")
print(f"Strategy outperformed benchmark: {outperformed}")

### Conclusion

**Strategy Viability:** The backtest results indicate whether the model-driven, forecast-informed portfolio allocation outperformed the passive 60/40 benchmark on a risk-adjusted basis. If the strategy achieves a higher Sharpe Ratio, it demonstrates that incorporating TSLA's forecasted return as an input to MPT optimization adds value beyond a simple static allocation. The higher TSLA weighting in the optimal portfolio amplifies returns during bullish periods but also increases drawdown exposure — a trade-off that is clearly visible in the drawdown comparison chart.

**Limitations of this Backtest:**
1. **Look-ahead bias risk** — The ARIMA model was trained on data up to end-2024; any information leakage into the forecast would inflate strategy performance.
2. **Single backtest window** — One year of backtesting is insufficient to draw statistically robust conclusions. A walk-forward or rolling-window backtest would provide stronger evidence.
3. **Transaction costs ignored** — Real-world rebalancing incurs brokerage fees, bid-ask spreads, and potential market impact, especially for TSLA's high-volume trading.
4. **Static covariance assumption** — The covariance matrix is computed on the full historical period; in practice, correlations shift during market stress events.
5. **EMH caveat** — Per the Efficient Market Hypothesis, consistent outperformance using historical price data alone is unlikely to persist out-of-sample. This backtest should be treated as a proof-of-concept rather than a production trading signal.

In [ ]:
# Save final metrics
metrics_df.to_csv(f'{PROCESSED_DIR}/backtest_metrics.csv')
print("Backtest metrics saved to data/processed/backtest_metrics.csv")